# Case 4, Part C: Feature Extraction and Comparison

This notebook splits the reviews into training and test sets, vectorizes the review text (`clean_text`), and compares **TF-IDF, bigrams, and a limited vocabulary**, with the rationale for each.


**Bottom line:** TF-IDF unigrams are the first choice, the top-500 vocabulary is a reasonable option if fewer dimensions are needed, and bigrams are not worth adding.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse, stats
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS,
    CountVectorizer,
    TfidfVectorizer,
)
from sklearn.feature_selection import chi2
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

DATA_PATH = Path("Case4_Labeled.csv")
OUT_DIR = Path("feature_extraction_results")
OUT_DIR.mkdir(exist_ok=True)

TEXT_COL, LABEL_COL, ID_COL, SPLIT_COL = (
    "clean_text",
    "final_label",
    "reviewer_id",
    "dataset_split",
)
SEED = 42
TEST_SIZE = 0.2
VOCAB_CAPS = [100, 200, 500]
N_SIGNAL_TERMS = 30
N_BOOTSTRAP = 300
BASELINE = "Baseline: count, unigram"

# Single-letter tokens are dropped, but stopwords are kept (see section 1).
TOKEN_PATTERN = r"(?u)\b\w\w+\b"

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_colwidth", None)

df = pd.read_csv(DATA_PATH)
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[LABEL_COL], random_state=SEED
)
train_texts, test_texts = train_df[TEXT_COL], test_df[TEXT_COL]
y_train = train_df[LABEL_COL].to_numpy()
print(f"Train: {len(train_df)} rows, {y_train.sum()} positive")
print(f"Test:  {len(test_df)} rows, {test_df[LABEL_COL].sum()} positive")

Train: 160 rows, 41 positive
Test:  40 rows, 10 positive


## 1. Design

These three methods are **not mutually exclusive**, because each controls a different aspect of the text representation. A unigram-count baseline is therefore used, with one factor changed at a time so that any difference can be attributed to that factor.

| Method | Factor controlled | Setting | Problem it addresses |
|---|---|---|---|
| Baseline | None | Counts, unigrams | Reference point |
| TF-IDF | Term weighting | TF-IDF, unigrams | Common words carry too much weight |
| Bigram | N-gram range | Counts, unigrams + bigrams | Unigrams lose word order and phrases |
| Limited vocabulary | Dimensionality | Counts, most frequent 100 / 200 / 500 terms | High dimensionality and sparsity |

**Why stopwords are kept:** sklearn's English stopword list removes words such as `not` and `too`, yet expressions like `too small` and `not true` relate directly to sizing and fit.

In [2]:
def build_feature_sets():
    """Return {name: unfitted vectorizer}. Each one changes a single factor."""
    features = {
        BASELINE: CountVectorizer(token_pattern=TOKEN_PATTERN),
        "TF-IDF, unigram": TfidfVectorizer(token_pattern=TOKEN_PATTERN),
        "Bigram: count, uni+bi": CountVectorizer(
            token_pattern=TOKEN_PATTERN, ngram_range=(1, 2)
        ),
    }
    for cap in VOCAB_CAPS:
        features[f"Limit vocab: count, top {cap}"] = CountVectorizer(
            token_pattern=TOKEN_PATTERN, max_features=cap
        )
    return features

## 2. Vectorize and export to CSV

The data is split first (stratified 80/20), and each vectorizer is then **fitted on the training set only and applied to the test set**. Fitting only on training data keeps test-set vocabulary and document frequencies out of the features. It also gives both sets the same columns. Test-set terms outside the training vocabulary are ignored, as they would be for a new review in production.

Each method is exported to one `vectors_<method>.csv` file. The first three columns are `reviewer_id`, `dataset_split` (`train` or `test`), and `final_label`; each remaining column is a term. Rows follow the order of the source CSV.

> **Note for Part D:** these matrices can be used directly for a final train/test evaluation. Cross-validation is different, because a vectorizer fitted on the full training set has already seen each validation fold. For tuning, refit the vectorizer inside each fold using a `Pipeline` and `build_feature_sets()`.

In [3]:
def save_method_csv(name, X_train, X_test, vocab):
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    assert not {ID_COL, SPLIT_COL, LABEL_COL} & set(vocab)
    vectors = pd.DataFrame(
        sparse.vstack([X_train, X_test]).toarray(),
        index=train_df.index.append(test_df.index),
        columns=vocab,
    ).sort_index()
    vectors.insert(0, LABEL_COL, df[LABEL_COL])
    vectors.insert(0, SPLIT_COL, "test")
    vectors.loc[train_df.index, SPLIT_COL] = "train"
    vectors.insert(0, ID_COL, df[ID_COL])
    path = OUT_DIR / f"vectors_{slug}.csv"
    vectors.to_csv(path, index=False, float_format="%.6f")
    return path


vectorizers, train_matrices, test_matrices, vocabularies = {}, {}, {}, {}
for name, vectorizer in build_feature_sets().items():
    # The vocabulary and IDF are learned from the training set only.
    train_matrices[name] = vectorizer.fit_transform(train_texts)
    test_matrices[name] = vectorizer.transform(test_texts)
    vectorizers[name] = vectorizer
    vocabularies[name] = vectorizer.get_feature_names_out()
    path = save_method_csv(
        name, train_matrices[name], test_matrices[name], vocabularies[name]
    )
    print(
        f"{path.name}: train {train_matrices[name].shape}, "
        f"test {test_matrices[name].shape}"
    )

vectors_baseline_count_unigram.csv: train (160, 970), test (40, 970)
vectors_tf_idf_unigram.csv: train (160, 970), test (40, 970)


vectors_bigram_count_uni_bi.csv: train (160, 3948), test (40, 3948)
vectors_limit_vocab_count_top_100.csv: train (160, 100), test (40, 100)
vectors_limit_vocab_count_top_200.csv: train (160, 200), test (40, 200)
vectors_limit_vocab_count_top_500.csv: train (160, 500), test (40, 500)


## 3. Comparison criteria

With no model involved, the features are assessed directly from five angles:

1. **Size and sparsity:** number of features, density, and terms per document.
2. **Test-set coverage:** the share of test-set tokens the training vocabulary cannot represent, and the number of test reviews left empty.
3. **Where the weight falls:** do the highest-weighted terms carry meaning, or are they words like `the` and `and`?
4. **Category signal retained:** are the terms most associated with the positive class still in the vocabulary?
5. **Similarity structure (pair AUC):** are same-category reviews more similar to each other than to reviews outside the category?

All of these use the training set only, except test-set coverage, which uses test-set text but not test-set labels.

### 3.1 Terms most associated with the category

Using the training set, terms are ranked by chi-squared, keeping only those that appear at a higher rate in the positive class.

In [4]:
def top_signal_terms(n=N_SIGNAL_TERMS, ngram_range=(1, 1)):
    vectorizer = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=ngram_range)
    present = (vectorizer.fit_transform(train_texts) > 0).astype(int)
    scores, _ = chi2(present, y_train)
    table = pd.DataFrame(
        {
            "term": vectorizer.get_feature_names_out(),
            "chi2": scores,
            "pos_docs": np.asarray(present[y_train == 1].sum(axis=0)).ravel(),
            "neg_docs": np.asarray(present[y_train == 0].sum(axis=0)).ravel(),
        }
    )
    pos_rate = table["pos_docs"] / (y_train == 1).sum()
    neg_rate = table["neg_docs"] / (y_train == 0).sum()
    table = table[pos_rate > neg_rate]
    return table.sort_values("chi2", ascending=False).head(n).reset_index(drop=True)


signal = top_signal_terms()
signal_bigrams = top_signal_terms(ngram_range=(2, 2))
signal.to_csv(OUT_DIR / "signal_unigrams.csv", index=False)
signal_bigrams.to_csv(OUT_DIR / "signal_bigrams.csv", index=False)

print("Unigrams")
display(signal.head(15))
print("Bigrams")
display(signal_bigrams.head(15))

Unigrams


,term,chi2,pos_docs,neg_docs
0,fit,58.049,20,0
1,size,37.732,13,0
2,perfectly,29.024,10,0
3,fits,17.415,6,0
4,sizing,17.415,6,0
5,small,14.512,5,0
6,sizes,11.610,4,0
7,medium,11.610,4,0
8,large,11.610,4,0
9,really,10.484,5,1


Bigrams


,term,chi2,pos_docs,neg_docs
0,fit perfectly,17.415,6,0
1,the size,11.610,4,0
2,for me,11.610,4,0
3,love them,11.610,4,0
4,new shirt,8.707,3,0
5,fit and,8.707,3,0
6,the fit,8.707,3,0
7,sent me,8.707,3,0
8,than the,8.707,3,0
9,perfectly and,8.707,3,0


**Observations:**
- `fit`, `size`, `perfectly`, and `sizing` appear 0 times in the negative class, so the category appears to be **fit and sizing**.
- Meaningful bigrams do exist (`fit perfectly`, `too small`), but they are mixed with combinations that carry little meaning, such as `for me` and `than the`.

### 3.2 Size, sparsity, and test-set coverage

In [5]:
def test_oov_share(vectorizer):
    """Share of test-set tokens that the training vocabulary cannot represent."""
    analyzer = vectorizer.build_analyzer()
    tokens = [token for text in test_texts for token in analyzer(text)]
    return 100 * np.mean([token not in vectorizer.vocabulary_ for token in tokens])


profile = pd.DataFrame(
    [
        {
            "features": name,
            "n_features": X.shape[1],
            "train_density_%": 100 * X.nnz / (X.shape[0] * X.shape[1]),
            "train_terms_per_doc": X.getnnz(axis=1).mean(),
            "stopword_share_%": 100
            * np.mean([term in ENGLISH_STOP_WORDS for term in vocabularies[name]]),
            f"signal_terms_kept_of_{N_SIGNAL_TERMS}": int(
                signal["term"].isin(vocabularies[name]).sum()
            ),
            "train_empty_docs": int((X.getnnz(axis=1) == 0).sum()),
            "test_empty_docs": int((test_matrices[name].getnnz(axis=1) == 0).sum()),
            "test_oov_tokens_%": test_oov_share(vectorizers[name]),
        }
        for name, X in train_matrices.items()
    ]
)
profile.to_csv(OUT_DIR / "feature_profile.csv", index=False)
display(profile)

,features,n_features,train_density_%,train_terms_per_doc,stopword_share_%,signal_terms_kept_of_30,train_empty_docs,test_empty_docs,test_oov_tokens_%
0,"Baseline: count, unigram",970,2.105,20.419,17.010,30,0,0,14.890
1,"TF-IDF, unigram",970,2.105,20.419,17.010,30,0,0,14.890
2,"Bigram: count, uni+bi",3948,1.102,43.513,4.179,30,0,0,41.760
3,"Limit vocab: count, top 100",100,11.281,11.281,59.000,14,3,0,40.441
4,"Limit vocab: count, top 200",200,7.128,14.256,45.500,23,0,0,29.412
5,"Limit vocab: count, top 500",500,3.496,17.481,26.000,30,0,0,20.680


**Observations:**
- The training set has only 970 unigrams, so the 10k–30k vocabulary suggested in the assignment does not apply here.
- Bigrams multiply the number of features by 4.1 and nearly halve the density.
- Bigrams also have the weakest test-set coverage: 41.8% of test-set unigram and bigram tokens are outside the training vocabulary, compared with 14.9% for the baseline. Many word pairs in new reviews were never seen in training.
- Smaller vocabularies have a higher stopword share (59% in the top 100), because `max_features` keeps terms by frequency, not by discriminative power.
- The top 500 keeps all 30 signal terms, while the top 200 loses 7 and the top 100 loses 16. Test-set out-of-vocabulary tokens rise from 14.9% for the full vocabulary to 20.7%, 29.4%, and 40.4% for the top 500, 200, and 100.
- No test review is left empty under any method, although the top 100 leaves 3 training reviews empty.

### 3.3 Where the weight falls

In [6]:
top_terms = pd.DataFrame(
    [
        {
            "features": name,
            "top_terms": ", ".join(
                vocabularies[name][
                    np.argsort(np.asarray(X.mean(axis=0)).ravel())[::-1][:10]
                ]
            ),
        }
        for name, X in train_matrices.items()
    ]
)
top_terms.to_csv(OUT_DIR / "top_weighted_terms.csv", index=False)
display(top_terms)

tfidf = vectorizers["TF-IDF, unigram"]
idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out())
display(idf[["the", "and", "great", "shirt", "fit", "size", "sizing"]].to_frame("idf"))

,features,top_terms
0,"Baseline: count, unigram","the, and, to, was, my, it, shirts, great, is, shirt"
1,"TF-IDF, unigram","the, and, great, quality, shirt, was, shirts, to, it, my"
2,"Bigram: count, uni+bi","the, and, to, was, my, it, shirts, great, is, shirt"
3,"Limit vocab: count, top 100","the, and, to, was, my, it, shirts, great, is, shirt"
4,"Limit vocab: count, top 200","the, and, to, was, my, it, shirts, great, is, shirt"
5,"Limit vocab: count, top 500","the, and, to, was, my, it, shirts, great, is, shirt"


,idf
the,1.712
and,1.615
great,2.150
shirt,2.320
fit,3.037
size,3.442
sizing,4.135


**Observations:** under raw counts, the top 10 terms are mostly words like `the`, `and`, and `to`. Under TF-IDF, `great` and `quality` move up; `the` and `and` still rank first and second, but with less influence. The IDF table also shows that `fit` and `size` have clearly higher weighting factors than `the` and `and`.

### 3.4 Similarity structure (pair AUC)

**Pair AUC** is the probability that a randomly chosen same-category pair of reviews has a higher cosine similarity than a randomly chosen cross-category pair; 0.5 means no separation. It does not depend on the absolute level of similarity, so it can be compared across methods. It is computed on the training set.

Confidence intervals come from a bootstrap over training reviews (300 resamples). All methods use the same resamples, so the differences are paired.

In [7]:
def pair_auc(similarity, idx):
    pos, neg = idx[y_train[idx] == 1], idx[y_train[idx] == 0]
    within = similarity[np.ix_(pos, pos)]
    # Upper triangle only, and never a document paired with its own resampled copy.
    keep = np.triu(pos[:, None] != pos[None, :], k=1)
    between = similarity[np.ix_(pos, neg)].ravel()
    result = stats.mannwhitneyu(within[keep], between)
    return result.statistic / (keep.sum() * between.size)


similarities = {name: cosine_similarity(X) for name, X in train_matrices.items()}
all_idx = np.arange(len(y_train))
rng = np.random.default_rng(SEED)
samples = [
    rng.choice(all_idx, size=len(y_train), replace=True) for _ in range(N_BOOTSTRAP)
]
boot = {
    name: np.array([pair_auc(sim, idx) for idx in samples])
    for name, sim in similarities.items()
}

rows = []
for name, sim in similarities.items():
    gap = boot[name] - boot[BASELINE]
    rows.append(
        {
            "features": name,
            "pair_auc": pair_auc(sim, all_idx),
            "gap_vs_baseline": gap.mean(),
            "gap_ci_low": np.percentile(gap, 2.5),
            "gap_ci_high": np.percentile(gap, 97.5),
        }
    )
separation = pd.DataFrame(rows)
separation.to_csv(OUT_DIR / "pair_auc_separation.csv", index=False)
display(separation)

,features,pair_auc,gap_vs_baseline,gap_ci_low,gap_ci_high
0,"Baseline: count, unigram",0.655,0.000,0.000,0.000
1,"TF-IDF, unigram",0.683,0.028,0.002,0.049
2,"Bigram: count, uni+bi",0.655,0.001,-0.003,0.005
3,"Limit vocab: count, top 100",0.646,-0.009,-0.018,0.001
4,"Limit vocab: count, top 200",0.647,-0.008,-0.015,-0.000
5,"Limit vocab: count, top 500",0.652,-0.003,-0.007,0.001


## 4. Comparison summary

| Method | Features | Train density | Test OOV tokens | Signal terms kept (of 30) | Pair AUC | Difference from baseline (95% CI) | Assessment |
|---|---|---|---|---|---|---|---|
| Baseline: counts, unigrams | 970 | 2.11% | 14.9% | 30 | 0.655 | – | Reference |
| **TF-IDF, unigrams** | 970 | 2.11% | 14.9% | 30 | **0.683** | +0.028 (+0.002, +0.049) | **First choice** |
| Bigram (uni+bi) | 3,948 | 1.10% | 41.8% | 30 | 0.655 | +0.001 (−0.003, +0.005) | Not worth adding |
| Limited vocabulary, top 500 | 500 | 3.50% | 20.7% | 30 | 0.652 | −0.003 (−0.007, +0.001) | Reasonable way to reduce dimensions |
| Limited vocabulary, top 200 | 200 | 7.13% | 29.4% | 23 | 0.647 | −0.008 (−0.015, −0.000) | Loses signal |
| Limited vocabulary, top 100 | 100 | 11.28% | 40.4% | 14 | 0.646 | −0.009 (−0.018, +0.001) | Loses signal |

OOV means out of vocabulary: test-set tokens that the training vocabulary cannot represent.

## 5. Rationale

### TF-IDF: reweighting terms
- **Idea:** lower the weight of terms that appear everywhere and raise the weight of rarer terms, with the vocabulary unchanged.
- **Result:** more weight falls on content words, and pair AUC is the highest of all methods.
- **Assessment:** it brings an improvement without adding dimensions or reducing test-set coverage, which makes it the most cost-effective change.

### Bigrams: keeping local word order
- **Idea:** capture phrases that single words cannot express, such as `fit perfectly` and `too small`.
- **Result:** the number of features grows 4.1 times and density nearly halves, while pair AUC barely changes. Test-set coverage is also the weakest.
- **Why:** useful phrases are few, the words inside them (`fit`, `small`) are already strong signals on their own, and noise such as `for me` and `than the` comes along with them.

### Limited vocabulary: controlling dimensionality
- **Idea:** keep only frequent terms to reduce sparsity and noise.
- **Result:** the top 500 removes 48% of the features, keeps every signal term, and shows no clear drop in pair AUC. Smaller vocabularies start losing signal terms.
- **Why:** `max_features` selects by frequency, not discriminative power, so the stopword share rises as the vocabulary shrinks.

## 6. Limitations

- **The differences are small:** the TF-IDF confidence interval excludes 0, but its lower bound is only +0.002 and the gain is 0.028, so it should be read as a modest improvement.
- **The sample is small:** the comparison uses 160 training reviews, 41 of them positive, so results could shift with a different split.
- **Pair AUC does not measure classification performance:** it describes similarity structure only. Classification performance can only be assessed in Part D.
- **The results are partly circular:** the signal terms and pair AUC both depend on `final_label`, and that label relies heavily on keywords such as `fit` and `size`.